# Lab 4: Deploy to a SageMaker Real-Time Endpoint

### Workshop notebooks

| # | Notebook | What you'll do |
|---|---|---|
| 1 | `1-prepare-data.ipynb` | Format ContractNLI for SFT and register the datasets in SageMaker |
| 2 | `2-fine-tune-llm.ipynb` | Launch the serverless LoRA fine-tuning job on Nemotron 3 Nano 30B |
| 3 | `3-evaluation.ipynb` | Score the base, fine-tuned, and frontier models |
| 4 | `4-deployment.ipynb` | Deploy the fine-tuned model to a SageMaker real-time endpoint |

---

This notebook deploys the fine-tuned Nemotron 3 Nano model to a SageMaker real-time endpoint and runs a serving check on a real contract.

Four AWS resources make up the serving stack:

| Resource | What it does |
|---|---|
| `Model` | Registers the merged checkpoint and the serving container |
| `EndpointConfig` | Defines the instance type and routing strategy |
| `Endpoint` | The always-on HTTPS API |
| `InferenceComponent` | Attaches the model to the endpoint and loads it onto the GPU |

Every create is wrapped in a `get`-first `try/except`, so all cells are safe to re-run: an existing resource is reused rather than duplicated.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker role arn: arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-ExecutionRole-20201215T102238
sagemaker bucket: sagemaker-us-east-1-492681118881
sagemaker session region: us-east-1


In [3]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so notebooks 2, 3, 4 and 4a all derive the
# same name for any model id.
MAX_MPG_NAME_LENGTH = 63
suffix = "-contractnli-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")


Model Package Group: huggingface-reasoning-nvidia-nemotro-b73b16-contractnli-sft-mpg


### Why the merged checkpoint, not the adapter

The training job registered a model package with two artifacts: the LoRA adapter on its own, and a **merged** checkpoint where the adapter weights have already been folded into the base model.

We deploy the merged checkpoint. A serving engine has to load the full model regardless; keeping the adapter separate adds complexity with no benefit. The cell below reads the package's `hf_merged` path straight from the registry.

In [4]:
from sagemaker.core import s3
from sagemaker.core.resources import ModelPackage

resp = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=1)
assert resp["ModelPackageSummaryList"], "no model packages found - run notebook 2 first"

model_package = ModelPackage.get(resp["ModelPackageSummaryList"][0]["ModelPackageArn"])

# the *merged* checkpoint (base weights + LoRA applied) is what we deploy
merged_model_s3_uri = s3.s3_path_join(
    model_package.inference_specification.containers[0]
    .model_data_source.s3_data_source.s3_uri,
    "checkpoints", "hf_merged") + "/"

print(f"merged model: {merged_model_s3_uri}")

[09/10/26 13:56:47] WARNING  No region provided. Using default region.                                 ]8;id=941009;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=989748;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

merged model: s3://sagemaker-us-east-1-492681118881/huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16-contractnli/contractnli-sft-20260910020026/output/model/checkpoints/hf_merged/


### Serving container

This notebook uses the AWS LMI (Large Model Inference) DJL container. Despite the different env-var prefix, the LMI container runs vLLM under the hood when `OPTION_ENTRYPOINT` points to `djl_python.lmi_vllm.vllm_async_service`. That's what the config below does.

Two things to know about the container pin:

- **Architecture support.** The container's bundled Transformers must recognise the model type. Older images reject Nemotron's architecture outright with *"Transformers does not recognize this architecture"*.
- **Driver compatibility.** Newer `cu130` images fail to start on `ml.g5` instances with `CannotStartContainerError`. The `cu128` pin below is verified working.

`OPTION_TENSOR_PARALLEL_DEGREE: max` lets vLLM use all GPUs on the instance. On `ml.g5.xlarge` that is one A10G; switching to a larger instance shards the model automatically.

In [5]:
import json

region = sess.boto_region_name
CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

instance_type = "ml.g5.xlarge"
health_check_timeout = 700

env = {
    "HF_MODEL_ID": "/opt/ml/model",
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 32),
}
print(inference_image)

763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.36.0-lmi18.0.0-cu128


### Resource names

SageMaker caps resource names at 63 characters, and the Nemotron model ID is long enough to exceed that once a suffix is added. The `rname` helper hash-truncates when needed, so any model ID produces a valid name. All four endpoint resources share the same stem, keeping them grouped together in the console.

In [6]:
import hashlib

MAX_NAME = 63


def rname(base, suffix):
    cand = f"{base}{suffix}"
    if len(cand) <= MAX_NAME:
        return cand
    digest = hashlib.sha1(base.encode()).hexdigest()[:6]
    keep = MAX_NAME - len(suffix) - len(digest) - 1
    return f"{base[:keep].rstrip('-')}-{digest}{suffix}"


stem = f"{base_model_id}-contractnli"
model_name = rname(stem, "-sft-m")
endpoint_config_name = rname(stem, "-sft-cfg")
endpoint_name = rname(stem, "-sft-ep")
ic_name = rname(stem, "-sft-ic")
print(model_name, endpoint_config_name, endpoint_name, ic_name, sep="\n")

huggingface-reasoning-nvidia-nemotron-3-nano-30b-a-674f44-sft-m
huggingface-reasoning-nvidia-nemotron-3-nano-30b-674f44-sft-cfg
huggingface-reasoning-nvidia-nemotron-3-nano-30b-674f44-sft-ep
huggingface-reasoning-nvidia-nemotron-3-nano-30b-674f44-sft-ic


### Create the Model, EndpointConfig, and Endpoint

These three resources set up the infrastructure. No weights are loaded yet.

- **`Model`** registers the merged S3 checkpoint with the LMI container and its configuration. It's a record in SageMaker. Nothing runs.
- **`EndpointConfig`** specifies the instance type and routing strategy. `LEAST_OUTSTANDING_REQUESTS` sends each new request to the replica with the shortest active queue.
- **`Endpoint`** provisions the compute. It comes up empty because we use an inference component to attach the model. The wait here is for the instance to boot, not for the model to load.

The endpoint takes 5-10 minutes to reach `InService`.

In [7]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes import (ContainerDefinition, ModelDataSource,
                                   ProductionVariant, S3ModelDataSource)

try:
    Model.get(model_name)
    print(f"model exists: {model_name}")
except Exception:
    Model.create(
        model_name=model_name,
        primary_container=ContainerDefinition(
            image=inference_image,
            model_data_source=ModelDataSource(
                s3_data_source=S3ModelDataSource(
                    s3_uri=merged_model_s3_uri, s3_data_type="S3Prefix",
                    compression_type="None")),
            environment=env),
        execution_role_arn=role)
    print(f"created model: {model_name}")

try:
    EndpointConfig.get(endpoint_config_name)
    print(f"endpoint config exists: {endpoint_config_name}")
except Exception:
    EndpointConfig.create(
        endpoint_config_name=endpoint_config_name,
        execution_role_arn=role,
        production_variants=[ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"})])
    print(f"created endpoint config: {endpoint_config_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[09/10/26 13:56:50] INFO     Creating model resource.                                            ]8;id=175046;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=302572;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#20592\20592]8;;\

created model: huggingface-reasoning-nvidia-nemotron-3-nano-30b-a-674f44-sft-m


[09/10/26 13:56:51] INFO     Creating endpoint_config resource.                                  ]8;id=60743;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=987792;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11068\11068]8;;\

created endpoint config: huggingface-reasoning-nvidia-nemotron-3-nano-30b-674f44-sft-cfg


In [8]:
try:
    endpoint = Endpoint.get(endpoint_name)
    print(f"endpoint exists: {endpoint_name}")
except Exception:
    endpoint = Endpoint.create(endpoint_name=endpoint_name,
                               endpoint_config_name=endpoint_config_name)
    print(f"creating endpoint: {endpoint_name}")

endpoint.wait_for_status("InService")
print("endpoint InService")

[09/10/26 13:56:52] INFO     Creating endpoint resource.                                         ]8;id=89757;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=996609;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#10227\10227]8;;\

Output()

creating endpoint: huggingface-reasoning-nvidia-nemotron-3-nano-30b-674f44-sft-ep


[09/10/26 14:06:06] INFO     Final Resource Status: InService                                    ]8;id=344645;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=243523;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#10483\10483]8;;\

endpoint InService


### Create the Inference Component

`InferenceComponent` is what actually loads the model weights onto the GPU. Separating it from the endpoint means you can attach multiple models to one endpoint, or run multiple copies of the same model for horizontal scaling, without reprovisioning the underlying instance.

`copy_count=1` loads one copy of the model. `number_of_accelerator_devices_required=1` claims the single A10G on `ml.g5.xlarge`. This wait is longer than the endpoint's: the merged checkpoint downloads from S3 and loads into GPU memory before the component reaches `InService`.

In [9]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (InferenceComponentComputeResourceRequirements,
                                   InferenceComponentRuntimeConfig,
                                   InferenceComponentSpecification)

try:
    ic = InferenceComponent.get(ic_name)
    print(f"inference component exists: {ic_name}")
except Exception:
    ic = InferenceComponent.create(
        inference_component_name=ic_name,
        endpoint_name=endpoint_name,
        variant_name="AllTraffic",
        specification=InferenceComponentSpecification(
            model_name=model_name,
            compute_resource_requirements=InferenceComponentComputeResourceRequirements(
                min_memory_required_in_mb=1024,
                number_of_accelerator_devices_required=1)),
        runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
        region=region)
    print(f"creating inference component: {ic_name}")

ic.wait_for_status("InService")

TUNED_MODEL_ID = f"sm:{endpoint_name}/{ic_name}@{region}"
print(f"\nready. serving model id:\n  {TUNED_MODEL_ID}")

                    INFO     Creating inference_component resource.                              ]8;id=213910;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=6399;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#16567\16567]8;;\

Output()

creating inference component: huggingface-reasoning-nvidia-nemotron-3-nano-30b-674f44-sft-ic


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:23                                                                                   │
│                                                                                                  │
│   20 │   │   region=region)                                                                      │
│   21 │   print(f"creating inference component: {ic_name}")                                       │
│   22                                                                                             │
│ ❱ 23 ic.wait_for_status("InService")                                                             │
│   24                                                                                             │
│   25 TUNED_MODEL_ID = f"sm:{endpoint_name}/{ic_name}@{region}"                                   │
│   26 print(f"\nready. serving model id:\n  {TUNED_MODEL_ID}")                                    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py:142 in wrapper               │
│                                                                                                  │
│     139 │   │   @functools.wraps(func)                                                           │
│     140 │   │   def wrapper(*args, **kwargs):                                                    │
│     141 │   │   │   config = dict(arbitrary_types_allowed=True)                                  │
│ ❱   142 │   │   │   return validate_call(config=config)(func)(*args, **kwargs)                   │
│     143 │   │                                                                                    │
│     144 │   │   return wrapper                                                                   │
│     145                                                                                          │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pydantic/_internal/_validate_call.py:39 in               │
│ wrapper_function                                                                                 │
│                                                                                                  │
│    36 │   │                                                                                      │
│    37 │   │   @functools.wraps(wrapped)                                                          │
│    38 │   │   def wrapper_function(*args, **kwargs):                                             │
│ ❱  39 │   │   │   return wrapper(*args, **kwargs)                                                │
│    40 │                                                                                          │
│    41 │   # We need to manually update this because `partial` object has no `__name__` and `__   │
│    42 │   wrapper_function.__name__ = extract_function_name(wrapped)                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pydantic/_internal/_validate_call.py:136 in __call__     │
│                                                                                                  │
│   133 │   │   if not self.__pydantic_complete__:                                                 │
│   134 │   │   │   self._create_validators()                                                      │
│   135 │   │                                                                                      │
│ ❱ 136 │   │   res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args,   │
│   137 │   │   if self.__return_pydantic_validator__:                                             │
│   138 │   │   │   return self.__return_pydantic_validator__

### Smoke test on a real contract

The request uses `C.build_messages(doc, labels)`, the same one every other notebook uses. Keeping the prompt in `contractnli.py` means the served request can't drift from the training data.

One thing to know: notebook 1 trains on `C.build_prompt`, a **single string** with the contract before the checklist. This endpoint expects **chat turns**, so the request switches to `C.build_messages`: checklist in the system turn, contract in the user turn. Same instruction and checklist, different order.

That's intentional. A chat endpoint wants a system turn, and this instruction is explicit enough to survive the reordering. The cell verifies it worked by checking that the served model still returns all 17 items as valid JSON. `temperature` is `0.0`: this is a classification task with a fixed output schema, and there's nothing to gain from sampling.

In [ ]:
import json
import re
import time

import boto3
from botocore.config import Config

import contractnli as C

C.ensure_dataset("./data")
test_docs, labels = C.load("test")
doc = test_docs[0]

# Raw SageMaker runtime call: this is the request an application would make.
# The LMI container speaks the OpenAI chat schema, and the inference component is
# selected with its own header, not inside the body.
smr = boto3.client(
    "sagemaker-runtime",
    region_name=region,
    config=Config(read_timeout=300, retries={"total_max_attempts": 3}),
)


def ask_endpoint(doc, max_tokens=4000):
    """Invoke the deployed model and return (text, usage).

    The turns come from `C.build_messages`, so the prompt is imported rather than
    rebuilt: pasting a second copy into a notebook is how train/serve skew gets
    introduced. Note that this is the two-turn form (checklist in the system turn), while
    notebook 1 trains on the single-string `C.build_prompt`, see the markdown above for
    why that is safe here and how to serve the trained string verbatim instead.
    """
    body = {
        "model_name": ic_name,
        "messages": [{"role": m["role"],
                      "content": [{"type": "text", "text": m["content"]}]}
                     for m in C.build_messages(doc, labels)],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stop": ["<|im_end|>"],
        "stream": False,
    }
    response = smr.invoke_endpoint(
        EndpointName=endpoint_name,
        InferenceComponentName=ic_name,
        ContentType="application/json",
        Body=json.dumps(body),
    )
    payload = json.loads(response["Body"].read())
    text = payload["choices"][0]["message"]["content"]
    usage = payload.get("usage", {})
    return text, usage


def parse_verdicts(text):
    """Minimal check that the served model returned a usable verdict object."""
    body = re.sub(r"<think>.*?</think>", " ", text or "", flags=re.DOTALL)
    fence = re.search(r"```(?:json)?\s*(.*?)```", body, re.DOTALL)
    if fence:
        body = fence.group(1)
    start, end = body.find("{"), body.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(body[start:end + 1])
    except json.JSONDecodeError:
        return None


t0 = time.time()
text, usage = ask_endpoint(doc)
elapsed = time.time() - t0

pred = parse_verdicts(text)
ok = pred is not None
print(f"first call {elapsed:.1f}s | valid JSON: {ok} | usage: {usage}")
print(f"answered {len(pred) if ok else 0} of {len(labels)} checklist items")

# Serving check only: no accuracy here. The base and fine-tuned model are scored
# exclusively by the managed evaluation pipeline in notebook 3, so the lab reports
# one set of numbers produced one way.


### Clean up

> **Important:** an endpoint bills per instance-hour for as long as it exists, whether or not you send traffic. An idle `ml.g5.xlarge` can cost more per day than the entire fine-tuning job. Delete it now.

The inference component must be deleted first; the endpoint refuses to delete while a component is still attached. The cell sleeps 45 seconds between the two to let the deletion propagate.

In [ ]:
import time

for label, fn in [
    ("inference component", lambda: InferenceComponent.get(ic_name).delete()),
    ("endpoint", lambda: Endpoint.get(endpoint_name).delete()),
    ("endpoint config", lambda: EndpointConfig.get(endpoint_config_name).delete()),
    ("model", lambda: Model.get(model_name).delete()),
]:
    try:
        fn()
        print(f"deleted {label}")
    except Exception as e:
        print(f"skip {label}: {type(e).__name__}")
    if label == "inference component":
        time.sleep(45)

---

## You're done!

You built the complete supervised fine-tuning pipeline on Amazon SageMaker, end to end:

1. **Lab 1**: Prepared the ContractNLI dataset and registered it in SageMaker AI Datasets
2. **Lab 2**: Fine-tuned Nemotron 3 Nano 30B with LoRA in a serverless job
3. **Lab 3**: Evaluated with statistical scoring and LLM-as-a-Judge
4. **Lab 4**: Deployed the merged model to a real-time endpoint and ran a serving check

Thanks for working through this workshop. We hope it gave you a practical feel for what serverless fine-tuning on SageMaker looks like on a real model and a real task.

Head to the [Summary](/05-summary/) page for a recap of what you built and ideas for what to explore next.